In [1]:
import pandas as pd
from pathlib import Path

In [ ]:
# 1. Define Paths Robustly (Works in both Jupyter and standard scripts)
current_dir = Path.cwd()

# Handle path depending on whether you run this from root or inside the 'scripts' folder
if current_dir.name == "scripts":
    root_dir = current_dir.parent
else:
    root_dir = current_dir

seller_data_dir = root_dir / 'data' / 'csvs'
seller_name = 'StarTech'
source_file_path = f"{seller_name}_products.csv"


In [ ]:
df = pd.read_csv(source_file_path)

# Remove any 'Unnamed' columns caused by trailing commas in the CSV
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Drop columns that are 100% empty
df = df.dropna(axis=1, how='all')

df_list.append(df)
print(f"Loaded {file_name} - Shape: {df.shape}")
else:
print(f"Warning: {file_name} not found at {file_path}")

Loaded clearance_sale.csv - Shape: (566, 6)
Loaded fragrance.csv - Shape: (1019, 6)
Loaded hair.csv - Shape: (3001, 6)
Loaded makeup.csv - Shape: (6442, 6)
Loaded personal_care.csv - Shape: (7029, 6)
Loaded skin.csv - Shape: (7146, 6)


In [197]:
# 4. Concatenate all loaded dataframes into one master dataframe
master_df = pd.concat(df_list, ignore_index=True)

In [198]:
# 5. Merge/Filter based on the uniqueness of the 'Link' field
# keep='first' retains the first instance it finds and drops subsequent duplicates
master_df = master_df.drop_duplicates(subset=['Link'], keep='first')
print(f"\nTotal unique products after dropping duplicate links: {len(master_df)}")


Total unique products after dropping duplicate links: 14060


In [199]:
# 6. Split the data based on missing values (NaNs)
# Rows with NO empty fields
cleaned_df = master_df.dropna(how='any')

In [200]:
# Rows with ANY empty fields
nan_df = master_df[master_df.isna().any(axis=1)]

In [201]:
# 7. Save to the respective files
cleaned_output_path = seller_data_dir / "shajgoj_cleaned_data.csv"
nan_output_path = seller_data_dir / "shajgoj_nan_rows.csv"

cleaned_df.to_csv(cleaned_output_path, index=False)
nan_df.to_csv(nan_output_path, index=False)

print(f"\nSaved successfully!")
print(f"Cleaned data (no empty fields) saved to: {cleaned_output_path.name} ({len(cleaned_df)} rows)")
print(f"NaN data (has empty fields) saved to: {nan_output_path.name} ({len(nan_df)} rows)")


Saved successfully!
Cleaned data (no empty fields) saved to: shajgoj_cleaned_data.csv (13995 rows)
NaN data (has empty fields) saved to: shajgoj_nan_rows.csv (65 rows)


In [202]:
# Load the template data 
template_df = pd.read_csv(template_file_path)
template_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   category_slug            0 non-null      object
 1   product_name             0 non-null      object
 2   product_slug             0 non-null      object
 3   seller_slug              0 non-null      object
 4   current_price            0 non-null      object
 5   seller_product_url       0 non-null      object
 6   brand_slug               0 non-null      object
 7   brand_name               0 non-null      object
 8   product_description      0 non-null      object
 9   model                    0 non-null      object
 10  sku                      0 non-null      object
 11  primary_image_url        0 non-null      object
 12  image_urls               0 non-null      object
 13  specifications           0 non-null      object
 14  attributes               0 non-null      object
 15  variation_ty

In [203]:
# Process the category fields by keeping only the first category before ';'
cleaned_df['Category'] = cleaned_df['Categories'].apply(lambda x: x.split(';')[0] if pd.notna(x) else x)
cleaned_df = cleaned_df.drop(columns=['Categories'])
print(cleaned_df[["Title", "Category"]].head())

# Show the unique categories after processing
unique_categories = cleaned_df['Category'].unique()
print(f"\nUnique categories after processing: {len(unique_categories)}")
print(unique_categories)


                                               Title    Category
0  Yc Baby Shampoo (Expiry Date: September,2026 )...  Mom & Baby
1  Isntree Hyaluronic Acid Airy Sun Stick (Expiry...        Skin
2  Yc Baby Bath gel (Expiry Date: July, 2028) 100...  Mom & Baby
3  Yc Baby Shampoo (Expiry Date: June  2027) 100....  Mom & Baby
4  Groome Acne Clear & Pore Minimizing Treatment ...        Skin

Unique categories after processing: 8
<StringArray>
[   'Mom & Baby',          'Skin',        'Makeup',          'Hair',
 'Personal care',           'Men',     'Fragrance',    'Appliances']
Length: 8, dtype: str


In [204]:
# Create a new category_slug column by following the dictionary mapping
category_mapping = {
    "Mom & Baby": "kids-care",
    "Skin": "skin-care",
    "Makeup": "makeup",
    "Hair": "haircare",
    "Personal Care": "personal-care",
    "Men": "mens-care",
    "Fragrance": "fragrance",
    "Appliances": "appliances",
}
cleaned_df['category_slug'] = cleaned_df['Category'].map(category_mapping)
print(cleaned_df[["Category", "category_slug"]].head())

     Category category_slug
0  Mom & Baby     kids-care
1        Skin     skin-care
2  Mom & Baby     kids-care
3  Mom & Baby     kids-care
4        Skin     skin-care


In [205]:
# Rename Title to product_name
cleaned_df = cleaned_df.rename(columns={"Title": "product_name"})
print(cleaned_df[["product_name", "category_slug"]].head())

                                        product_name category_slug
0  Yc Baby Shampoo (Expiry Date: September,2026 )...     kids-care
1  Isntree Hyaluronic Acid Airy Sun Stick (Expiry...     skin-care
2  Yc Baby Bath gel (Expiry Date: July, 2028) 100...     kids-care
3  Yc Baby Shampoo (Expiry Date: June  2027) 100....     kids-care
4  Groome Acne Clear & Pore Minimizing Treatment ...     skin-care


In [ ]:
# Add seller slug column and populate with the seller slug
cleaned_df['seller_slug'] = seller_name
print(cleaned_df[["product_name", "category_slug", "seller_slug"]].head())

                                        product_name category_slug seller_slug
0  Yc Baby Shampoo (Expiry Date: September,2026 )...     kids-care     shajgoj
1  Isntree Hyaluronic Acid Airy Sun Stick (Expiry...     skin-care     shajgoj
2  Yc Baby Bath gel (Expiry Date: July, 2028) 100...     kids-care     shajgoj
3  Yc Baby Shampoo (Expiry Date: June  2027) 100....     kids-care     shajgoj
4  Groome Acne Clear & Pore Minimizing Treatment ...     skin-care     shajgoj


In [207]:
# Rename the 'Sale Price' column to 'current_price' and 'Price' column to 'original_price'  
cleaned_df = cleaned_df.rename(columns={"Sale Price": "current_price", "Price": "original_price"})
cleaned_df.head()


,product_name,current_price,Link,original_price,Brands,Category,category_slug,seller_slug
0,"Yc Baby Shampoo (Expiry Date: September,2026 )...",450.0,https://shop.shajgoj.com/product/yc-baby-shamp...,450.0,YC,Mom & Baby,kids-care,shajgoj
1,Isntree Hyaluronic Acid Airy Sun Stick (Expiry...,299.0,https://shop.shajgoj.com/product/isntree-hyalu...,2250.0,Isntree,Skin,skin-care,shajgoj
2,"Yc Baby Bath gel (Expiry Date: July, 2028) 100...",260.0,https://shop.shajgoj.com/product/yc-baby-bath-...,260.0,YC,Mom & Baby,kids-care,shajgoj
3,Yc Baby Shampoo (Expiry Date: June 2027) 100....,300.0,https://shop.shajgoj.com/product/yc-baby-shamp...,300.0,YC,Mom & Baby,kids-care,shajgoj
4,Groome Acne Clear & Pore Minimizing Treatment ...,99.0,https://shop.shajgoj.com/product/groome-acne-c...,495.0,Groome,Skin,skin-care,shajgoj


In [208]:
# Rename Brands to 'brand_name'
cleaned_df = cleaned_df.rename(columns={"Brands": "brand_name"})
cleaned_df.head()

,product_name,current_price,Link,original_price,brand_name,Category,category_slug,seller_slug
0,"Yc Baby Shampoo (Expiry Date: September,2026 )...",450.0,https://shop.shajgoj.com/product/yc-baby-shamp...,450.0,YC,Mom & Baby,kids-care,shajgoj
1,Isntree Hyaluronic Acid Airy Sun Stick (Expiry...,299.0,https://shop.shajgoj.com/product/isntree-hyalu...,2250.0,Isntree,Skin,skin-care,shajgoj
2,"Yc Baby Bath gel (Expiry Date: July, 2028) 100...",260.0,https://shop.shajgoj.com/product/yc-baby-bath-...,260.0,YC,Mom & Baby,kids-care,shajgoj
3,Yc Baby Shampoo (Expiry Date: June 2027) 100....,300.0,https://shop.shajgoj.com/product/yc-baby-shamp...,300.0,YC,Mom & Baby,kids-care,shajgoj
4,Groome Acne Clear & Pore Minimizing Treatment ...,99.0,https://shop.shajgoj.com/product/groome-acne-c...,495.0,Groome,Skin,skin-care,shajgoj


In [209]:
# Create Brand Slug by converting brand_name to lowercase and replacing spaces with hyphens
cleaned_df['brand_slug'] = cleaned_df['brand_name'].str.lower().str.replace(' ', '-').str.replace(".", "")
print(cleaned_df[["brand_name", "brand_slug"]].head(40))

       brand_name     brand_slug
0              YC             yc
1         Isntree        isntree
2              YC             yc
3              YC             yc
4          Groome         groome
5   Nirvana Color  nirvana-color
6   Nirvana Color  nirvana-color
7   Nirvana Color  nirvana-color
8   Nirvana Color  nirvana-color
9           LILAC          lilac
10         Pretty         pretty
11         Simple         simple
12     White Tone     white-tone
13      skin cafe      skin-cafe
14     Dr. Clinic      dr-clinic
15  Nirvana Color  nirvana-color
16          LILAC          lilac
17  Nirvana Color  nirvana-color
18      skin cafe      skin-cafe
19  Nirvana Color  nirvana-color
20  Nirvana Color  nirvana-color
24            XHC            xhc
25      skin cafe      skin-cafe
27      skin cafe      skin-cafe
28  Nirvana Color  nirvana-color
29  Nirvana Color  nirvana-color
30         Lavino         lavino
31          Ossum          ossum
32          Ossum          ossum
33      sk

In [210]:
# Rename "Link" to "seller_product_url", "Category" to "category_name"
cleaned_df = cleaned_df.rename(columns={"Link": "seller_product_url", "Category": "category_name"})
cleaned_df.head()

,product_name,current_price,seller_product_url,original_price,brand_name,category_name,category_slug,seller_slug,brand_slug
0,"Yc Baby Shampoo (Expiry Date: September,2026 )...",450.0,https://shop.shajgoj.com/product/yc-baby-shamp...,450.0,YC,Mom & Baby,kids-care,shajgoj,yc
1,Isntree Hyaluronic Acid Airy Sun Stick (Expiry...,299.0,https://shop.shajgoj.com/product/isntree-hyalu...,2250.0,Isntree,Skin,skin-care,shajgoj,isntree
2,"Yc Baby Bath gel (Expiry Date: July, 2028) 100...",260.0,https://shop.shajgoj.com/product/yc-baby-bath-...,260.0,YC,Mom & Baby,kids-care,shajgoj,yc
3,Yc Baby Shampoo (Expiry Date: June 2027) 100....,300.0,https://shop.shajgoj.com/product/yc-baby-shamp...,300.0,YC,Mom & Baby,kids-care,shajgoj,yc
4,Groome Acne Clear & Pore Minimizing Treatment ...,99.0,https://shop.shajgoj.com/product/groome-acne-c...,495.0,Groome,Skin,skin-care,shajgoj,groome


In [211]:
# Add product_slug by converting the "seller_product_url"'s last part after the last '/' and removing query parameters
cleaned_df['product_slug'] = cleaned_df['seller_product_url'].apply(lambda x: x.split('/')[-1].split('?')[0] if pd.notna(x) else x)
cleaned_df.head()

,product_name,current_price,seller_product_url,original_price,brand_name,category_name,category_slug,seller_slug,brand_slug,product_slug
0,"Yc Baby Shampoo (Expiry Date: September,2026 )...",450.0,https://shop.shajgoj.com/product/yc-baby-shamp...,450.0,YC,Mom & Baby,kids-care,shajgoj,yc,yc-baby-shampoo-expiry-date-september2026-2000-ml
1,Isntree Hyaluronic Acid Airy Sun Stick (Expiry...,299.0,https://shop.shajgoj.com/product/isntree-hyalu...,2250.0,Isntree,Skin,skin-care,shajgoj,isntree,isntree-hyaluronic-acid-airy-sun-stick-expiry-...
2,"Yc Baby Bath gel (Expiry Date: July, 2028) 100...",260.0,https://shop.shajgoj.com/product/yc-baby-bath-...,260.0,YC,Mom & Baby,kids-care,shajgoj,yc,yc-baby-bath-gel-expiry-date-july-2028-1000-ml
3,Yc Baby Shampoo (Expiry Date: June 2027) 100....,300.0,https://shop.shajgoj.com/product/yc-baby-shamp...,300.0,YC,Mom & Baby,kids-care,shajgoj,yc,yc-baby-shampoo-expiry-date-june-2027-1000-ml
4,Groome Acne Clear & Pore Minimizing Treatment ...,99.0,https://shop.shajgoj.com/product/groome-acne-c...,495.0,Groome,Skin,skin-care,shajgoj,groome,groome-acne-clear-pore-minimizing-treatment-ex...


,category_slug,product_name,product_slug,seller_slug,current_price,seller_product_url,brand_slug,brand_name,product_description,model,...,estimated_delivery_days,seller_sku,seller_product_name,category_path,category_name,category_description,seller_name,base_url,seller_country_code,is_active
